# Notebook 3 — Isotypic Decomposition and Lie Accessibility

Beyond the 9 primitive sectors lies the **isotypic decomposition** — 51 irreducible components resolved by the full commutant algebra. We also compute the Lie generators $A_g = \log\rho(g)$ and the $\kappa_d$ hierarchy to reveal the discrete/continuous split.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from rime.cubieoperator import CubieSpectralOperator, eigenspaces
from rime.cubie import CubieMove
from rime.cubieworld import SlowDynamics
from rime import helpers
from rime.spectral_utils import joint_diag_sectors, compute_transport_kappa
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
op = CubieSpectralOperator.from_gens_dict(CubieMove.prim_moves)
layers = sorted(op._layers, reverse=True)
print(f'Operator built: {len(layers)} spectral layers, dims={[op._layer_dim(lam) for lam in layers]}')
from rime.spectral_utils import find_t7_pairs, analyze_t7
import sys; sys.path.insert(0, os.path.abspath('.'))
from experiments.isotypic_decomposition import (
    isotypic_decomposition, isotypic_transport_tensor,
    multiplicity_transport, analyze_multiplicity_transport
)

## 2. Isotypic Decomposition — 51 Components

The commutant $\mathrm{Comm}_G(V_\lambda)$ is computed combinatorially within each spectral layer. Its center yields the isotypic decomposition. Result: **51 isotypic components**, of which **50 are multiplicity-free** (multiplicity $m=1$).

In [ ]:
iso = isotypic_decomposition(op)
print(f'Total isotypic components: {sum(b["center_dim"] for b in iso["blocks"].values())}')
print()
for lam in sorted(iso['blocks'], reverse=True):
    blk = iso['blocks'][lam]
    components = '  '.join(f'{d}D×{m}' for d, m in blk['isotypic'])
    print(f'  V_{lam:.4f} ({blk["dim"]:>3d}d):  {components}')

## 3. Multiplicity Histogram

The stark distribution: $m=1$ for 50 components, $m=11$ for exactly one. The representation is **almost multiplicity-free** — all internal multiplicity is concentrated into a single reservoir.

In [ ]:
from collections import Counter
mults = []
for lam in sorted(iso['blocks'], reverse=True):
    for d, m in iso['blocks'][lam]['isotypic']:
        mults.append(m)

counts = Counter(mults)
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
m_vals = sorted(counts.keys())
colors = ['#D62828' if m == 11 else '#2E86AB' for m in m_vals]
bars = ax.bar([str(m) for m in m_vals], [counts[m] for m in m_vals],
              color=colors, edgecolor='#333', linewidth=1.2)
for m, c in zip(m_vals, [counts[m] for m in m_vals]):
    ax.text(str(m), c + 0.5, str(c), ha='center', fontsize=14, fontweight='bold',
            color='#D62828' if m == 11 else '#2E86AB')
ax.set_xlabel('Multiplicity $m$', fontsize=13)
ax.set_ylabel('Number of Isotypic Components', fontsize=13)
ax.set_title('Almost Multiplicity-Free: One Reservoir at $m=11$', fontsize=14)
ax.set_yscale('log'); ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

## 4. The Multiplicity Reservoir

The unique component with $m>1$ is $V_{5/9}^{(3,11)}$ — 33 dimensions within the 106-dim $V_{5/9}$ layer. Its $11\times 11$ multiplicity transfer matrix has **full rank** — all 11 copies are independently active.

In [ ]:
mt_result = multiplicity_transport(op, seed=42)
diag_rows = analyze_multiplicity_transport(mt_result)

# Find the reservoir
for r in diag_rows:
    if r['eff_rank'] > 1:
        print(f'Reservoir: {r["pair"]}')
        print(f'  Shape: {r["shape"]},  max K: {r["max_K"]:.3f}')
        print(f'  Effective rank: {r["eff_rank"]},  Entropy: {r["entropy"]:.3f}')
        print(f'  Isotropy: {r["isotropy"]:.3f}')
        sv = r['svals'][:r['eff_rank']]
        print(f'  Singular values: {[f"{s:.3f}" for s in sv]}')

        # SVD spectrum
        fig, ax = plt.subplots(1, 1, figsize=(7, 4))
        ax.bar(range(1, len(sv)+1), sv/sv[0], color=plt.cm.inferno(np.linspace(0.2, 0.9, len(sv))),
               edgecolor='#222', linewidth=1)
        ax.plot(range(1, len(sv)+1), sv/sv[0], 'o-', color='#D62828', markersize=8)
        ax.set_xlabel('Channel index', fontsize=12)
        ax.set_ylabel('Normalized singular value', fontsize=12)
        ax.set_title('Reservoir Internal Channel Hierarchy', fontsize=14)
        ax.set_xticks(range(1, len(sv)+1))
        ax.grid(axis='y', alpha=0.2)
        plt.tight_layout(); plt.show()

## 5. Lie Generators and the $\kappa_d$ Hierarchy

The Lie generator $A_g = \log\rho(g)$ embeds the discrete group action into a continuous one-parameter subgroup. The transport at Lie depth $d$ is $\kappa_d(i,j) = \max\|P_i C_d P_j\|_F$ where $C_d$ ranges over all depth-$d$ nested commutators of the $A_g$.

Key result: $\kappa_1 > 0$ for some pairs with $\kappa_0 \approx 0$ — curvature creates new transport channels that gradient flow cannot access.

In [ ]:
# Compute Lie generators and κ at depths 0, 1, 2
op_lite = CubieSpectralOperator.from_gens_dict(CubieMove.prim_moves)

# κ₀ — gradient-level
kappa0 = np.zeros((6, 6))
# κ₁ — curvature-level
kappa1 = np.zeros((6, 6))

for i, li in enumerate(layers):
    Pi = op_lite.eigenspace_projector(li)
    for j, lj in enumerate(layers):
        Pj = op_lite.eigenspace_projector(lj)
        max_k0 = 0; max_k1 = 0
        for key, (mv, rho, ag, *_) in op_lite.rho_moves.items():
            Pi_ag = Pi @ ag
            Pi_ag_Pj = Pi_ag @ Pj
            max_k0 = max(max_k0, np.linalg.norm(Pi_ag_Pj, 'fro'))
            # κ₁ via commutator [A_g, A_h]
            for key2, (mv2, rho2, ah, *_) in op_lite.rho_moves.items():
                comm = ag @ ah - ah @ ag
                max_k1 = max(max_k1, np.linalg.norm(Pi @ comm @ Pj, 'fro'))
        kappa0[i, j] = max_k0
        kappa1[i, j] = max_k1

print(f'{kappa0.shape=}')
# Show the enhancement ratio κ₁/κ₀ for the V₇/₉↔V₂/₃ pair
i_79 = layers.index(7/9)
i_23 = layers.index(2/3)
print(f'κ₀(V₇/₉, V₂/₃) = {kappa0[i_79, i_23]:.2e}')
print(f'κ₁(V₇/₉, V₂/₃) = {kappa1[i_79, i_23]:.2f}')
print(f'Enhancement ratio κ₁/κ₀ ≈ {kappa1[i_79, i_23]/kappa0[i_79, i_23]:.0e}')
print('Curvature (commutators) is the dominant coupling mechanism for this pair.')

## 6. T7 Minimal Prototype: $S_3$ Natural ⊕ Regular

The T7 mechanism has a minimal 9-dimensional prototype: $S_3$ with block-diagonal representation $\rho = \mathrm{nat}(3) \oplus \mathrm{reg}(6)$. This system has 5 sectors, 1 hybrid bridge, and 3 composition-only T7 pairs — proving that T7 requires neither $M_2(\mathbb{C})$ nor noncommutativity.

In [ ]:
from rime.spectral_utils import build_s3_nat_rep, build_s3_regular_rep, build_block_diag_rho

# Build S₃ nat⊕reg (9-dim)
nat = build_s3_nat_rep()
reg = build_s3_regular_rep()
rho_s3, key_list_s3 = build_block_diag_rho([nat, reg])

# Compute A = average of all S₃ elements
n_gens = len(rho_s3)
A_s3 = sum(r for r in rho_s3) / n_gens

# Diagonalize
evals, evecs = np.linalg.eigh(A_s3)
print('S₃ nat⊕reg spectrum:')
for ev in sorted(set(np.round(evals, 10)), reverse=True):
    d = int(np.sum(np.abs(evals - ev) < 1e-8))
    print(f'  λ = {ev:.6f}  dim = {d}')

# Find T7 pairs
kappa = compute_transport_kappa(A_s3, rho_s3, tol=1e-10)
# T7 pairs cross the block boundary (indices 0-2 = nat, 3-8 = reg)
t7_result = find_t7_pairs(kappa, block_slices=[slice(0,3), slice(3,9)], tol=1e-10)
if t7_result:
    print(f'\n{t7_result["n_t7"]} T7 pairs found (S₃ nat⊕reg)')
    for pair in t7_result['pairs'][:5]:
        print(f'  {pair}')
else:
    print('\nNo T7 pairs — check block slice definitions')

## Summary

| Concept | Notebook | Paper |
|---------|----------|-------|
| Spectral layers (6) | NB1 | I |
| Rational spectral law | NB1 | I |
| Primitive sectors (9) | NB2 | I/II |
| Transport tensor $K_{ij}$ | NB2 | II |
| T7 pairs | NB2, NB3 | III |
| Isotypic decomposition (51) | NB3 | Appendix B |
| Multiplicity reservoir | NB3 | Appendix B |
| $\kappa_d$ Lie hierarchy | NB3 | III |
| $S_3$ minimal T7 prototype | NB3 | III |